## Data Collections

In [ ]:
import kaggle
import pandas as pd
import os
import csv

In [ ]:
# Télécharger le dataset depuis Kaggle
# !kaggle datasets download -d rishabhkausish/reddit-depression-dataset -p ../data/ --unzip

Dataset URL: https://www.kaggle.com/datasets/rishabhkausish/reddit-depression-dataset
License(s): CC0-1.0
100%|███████████████████████████████████████▉| 430M/431M [00:10<00:00, 40.6MB/s]
100%|████████████████████████████████████████| 431M/431M [00:10<00:00, 44.8MB/s]


In [6]:
# Chercher le fichier CSV extrait
csv_files = [f for f in os.listdir("../data/") if f.endswith('.csv')]
if csv_files:
    df = pd.read_csv(os.path.join("../data/", csv_files[0]))
    print("Dataset chargé dans le DataFrame 'df' est:", csv_files[0])
else:
    print("Aucun fichier CSV trouvé dans le dossier ../data/.")

/tmp/ipykernel_5266/3439755198.py:4: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(os.path.join("../data/", csv_files[0]))


Dataset chargé dans le DataFrame 'df' est: reddit_depression_dataset.csv


In [7]:
# Supprimer les lignes vides de la colonne Unnamed: 0
print("Nombre de valeurs manquantes dans la colonne 'Unnamed: 0' avant suppression :", df["Unnamed: 0"].isna().sum())
df = df.dropna(subset=["Unnamed: 0"])
print("Nombre de valeurs manquantes dans la colonne 'Unnamed: 0' après suppression :", df["Unnamed: 0"].isna().sum())

Nombre de valeurs manquantes dans la colonne 'Unnamed: 0' avant suppression : 3
Nombre de valeurs manquantes dans la colonne 'Unnamed: 0' après suppression : 0


In [8]:
# Supprimer les lignes vides de la colonne title
print("Nombre de valeurs manquantes dans la colonne 'title' avant suppression :", df["title"].isna().sum())
df = df.dropna(subset=["title"])
print("Nombre de valeurs manquantes dans la colonne 'title' après suppression :", df["title"].isna().sum())

Nombre de valeurs manquantes dans la colonne 'title' avant suppression : 23
Nombre de valeurs manquantes dans la colonne 'title' après suppression : 0


In [9]:
# 1) Copie de sécurité
df_backup = df.copy(deep=True)

# 2) Lignes où title contient au moins un chiffre
# mask = df["title"].astype(str).str.contains(r"\d", na=False) # Vérifie si la colonne "title" contient au moins un chiffre
unnamed_is_numeric = pd.to_numeric(df["Unnamed: 0"], errors="coerce").notna() # Vérifie si la colonne "Unnamed: 0" est numérique
title_is_numeric = pd.to_numeric(df["title"], errors="coerce").notna() # Vérifie si la colonne "title" est numérique
mask = (~unnamed_is_numeric) & title_is_numeric # On veut les lignes où "Unnamed: 0" n'est pas numérique et "title" est numérique

# 3) Inversion des valeurs entre les 2 colonnes sur ces lignes
df.loc[mask, ["Unnamed: 0", "title"]] = (
    df.loc[mask, ["title", "Unnamed: 0"]].to_numpy()
)

In [10]:
# 1) Vérifier que Unnamed: 0 est bien numérique partout
print(f"Nombre de valeurs non numériques dans la colonne 'Unnamed: 0' : {pd.to_numeric(df['Unnamed: 0'], errors='coerce').isna().sum()}")

# 2) Vérifier que title est bien majoritairement non numérique
print(f"Nombre de titres numériques dans la colonne 'title' : {pd.to_numeric(df['title'], errors='coerce').notna().sum()}")

# 3) Si tout est ok, convertir définitivement Unnamed: 0 en int
# df["Unnamed: 0"] = df["Unnamed: 0"].astype(int)
df["Unnamed: 0"] = pd.to_numeric(df["Unnamed: 0"], errors="coerce").astype("Int64")

# 4) Rename the column "Unnamed: 0" to "id"
df = df.rename(columns={"Unnamed: 0": "id"})

# 5) Set the "id" column as the index of the DataFrame
df = df.set_index("id")

Nombre de valeurs non numériques dans la colonne 'Unnamed: 0' : 0
Nombre de titres numériques dans la colonne 'title' : 212


In [11]:
# Supprime les lignes vides de la colonne "label"
print(f"Nombre de valeurs manquantes dans 'label' avant suppression : {df['label'].isna().sum()}") # Affiche le nombre de valeurs manquantes dans la colonne "label" avant suppression
df = df.dropna(subset=["label"])
print(f"Nombre de valeurs manquantes dans 'label' après suppression : {df['label'].isna().sum()}") # Affiche le nombre de valeurs manquantes dans la colonne "label" après suppression

Nombre de valeurs manquantes dans 'label' avant suppression : 83
Nombre de valeurs manquantes dans 'label' après suppression : 0


In [12]:
# Enregistrer le DataFrame nettoyé dans un nouveau fichier CSV
df.to_csv("../data/reddit_depression_dataset_cleaned_v1.csv", index=False)

In [2]:
# Ouvre le CSV ../data/reddit_depression_dataset_cleaned_v1.csv
df = pd.read_csv("../data/reddit_depression_dataset_cleaned_v1.csv")

Points forts :

Recall 91% sur dépressifs : détecte 9/10 posts dépressifs (crucial en santé mentale)

Precision 76% : quand il dit "dépressif", c’est juste 3 fois sur 4

F1 83% : excellent compromis

# Données propres.
Pipeline de préparation pour NLP :

Étape 1 : EDA rapide

In [3]:
# Distribution des labels
print(f"Distribution des labels :")
print(df["label"].value_counts(normalize=True) * 100)

Distribution des labels :
label
0.0    80.555509
1.0    19.444491
Name: proportion, dtype: float64


In [4]:
# Longueur des textes
df.loc[:, "text_length"] = df["title"].str.len() + df["body"].str.len()
print("Longueur textes :\n", df["text_length"].describe())

Longueur textes :
 count    2.009641e+06
mean     5.077977e+02
std      1.005511e+03
min      2.000000e+00
25%      1.030000e+02
50%      2.090000e+02
75%      5.330000e+02
max      6.485800e+04
Name: text_length, dtype: float64


In [5]:
# NaN restants
print(f"Nombre de valeurs manquantes dans le DataFrame : {df.isna().sum().sum()}")

Nombre de valeurs manquantes dans le DataFrame : 1035926


In [4]:
# Concat title + body (standard pour Reddit)
# df = df.assign(text=lambda x: x["title"].fillna("") + " " + x["body"].fillna(""))
df.loc[:, "clean_text"] = df["title"].fillna("") + " " + df["body"].fillna("")

# Nettoyage basique
import re
def clean_text(text):
    if pd.isna(text):
        return ""
    text = re.sub(r'http\S+', '', str(text)) # Remove URLs
    text = re.sub(r'[^a-zA-Z\s]', '', str(text)) # Remove punctuation and numbers
    return text.lower().strip() # Convert to lowercase and remove leading/trailing whitespace

df.loc[:, "clean_text"] = df["clean_text"].apply(clean_text)

In [ ]:
# Drop "title" and "body" columns
df = df.drop(columns=["title", "body"])

In [ ]:
# Save the cleaned DataFrame to a new CSV file
df.to_csv("../data/reddit_depression_dataset_cleaned_v2.csv", index=False, quoting=csv.QUOTE_ALL)
print("✅ Dataset nettoyé sauvegardé !\nConcat title + body et nettoyage basique effectué dans la colonne 'clean_text'.")

✅ Dataset nettoyé sauvegardé !
Concat title + body et nettoyage basique effectué dans la colonne 'clean_text'.


In [6]:
df.head()

,subreddit,title,body,upvotes,created_utc,num_comments,label,clean_text
0,DeepThoughts,Deep thoughts underdog,"Only when we start considering ourselves, the ...",4.0,1.405309e+09,NaN,0.0,deep thoughts underdog only when we start cons...
1,DeepThoughts,"I like this sub, there's only two posts yet I ...",Anyway: Human Morality is a joke so long as th...,4.0,1.410568e+09,1.0,0.0,i like this sub theres only two posts yet i ke...
2,DeepThoughts,Rebirth!,Hello. \nI am the new guy in charge here (Besi...,6.0,1.416458e+09,1.0,0.0,rebirth hello \ni am the new guy in charge her...
3,DeepThoughts,"""I want to be like water. I want to slip throu...",NaN,25.0,1.416512e+09,2.0,0.0,i want to be like water i want to slip through...
4,DeepThoughts,Who am I?,You could take any one cell in my body and kil...,5.0,1.416516e+09,4.0,0.0,who am i you could take any one cell in my bod...


In [ ]:
# 4. Sauvegarde
# df[["label", "clean_text"]].to_parquet("reddit_depression_clean.parquet")
# print("✅ Dataset nettoyé sauvegardé !")

✅ Dataset nettoyé sauvegardé !


In [22]:
print(f"Distribution des labels :")
print(df["label"].value_counts())

Distribution des labels :
label
0.0    1990260
1.0     480409
Name: count, dtype: int64


# Dataset prêt avec ~2M posts non-dépressifs (80,8%) et ~480k dépressifs (19,2%).

1. Diagnostic du déséquilibre
Problème majeur : classe très déséquilibrée (80/20). Un modèle naïf qui prédit toujours "0" aurait déjà 80% d’accuracy.
​
On doit absolument gérer ça pour avoir un modèle utile

2. Premier modèle de base (BC02)
Voici le code complet d’entraînement d’un modèle baseline simple qui gère le déséquilibre :

In [23]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
import joblib

# 1. Split stratifié (même répartition train/test)
X = df["clean_text"]
y = df["label"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
# 2. Vectorisation TF-IDF (5000 features, bi-grammes)
vectorizer = TfidfVectorizer(
    max_features=5000,
    stop_words='english',
    ngram_range=(1,2),
    min_df=5
)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

In [25]:
# 3. Modèle avec gestion déséquilibre (class_weight)
model = LogisticRegression(
    random_state=42, 
    max_iter=1000,
    class_weight='balanced'  # ⚠️ CLÉ pour le déséquilibre
)
model.fit(X_train_tfidf, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:

In [27]:
# 4. Évaluation
y_pred = model.predict(X_test_tfidf)
print("Classification Report :\n", classification_report(y_test, y_pred))
print("Confusion Matrix :\n", confusion_matrix(y_test, y_pred))

Classification Report :
               precision    recall  f1-score   support

         0.0       0.98      0.93      0.95    398052
         1.0       0.76      0.91      0.83     96082

    accuracy                           0.93    494134
   macro avg       0.87      0.92      0.89    494134
weighted avg       0.94      0.93      0.93    494134

Confusion Matrix :
 [[370326  27726]
 [  8253  87829]]


In [28]:
# 5. Sauvegarde (pour RNCP BC02)
joblib.dump(model, '../models/depression_classifier.pkl')
joblib.dump(vectorizer, '../models/tfidf_vectorizer.pkl')
print("✅ Modèle baseline sauvegardé !")

✅ Modèle baseline sauvegardé !


In [29]:
# Charger et tester
loaded_model = joblib.load('../models/depression_classifier.pkl')
loaded_vectorizer = joblib.load('../models/tfidf_vectorizer.pkl')

In [30]:
test_text = "I feel very unhappy and hopeless about my life"  # Exemple de texte à tester
test_vec = loaded_vectorizer.transform([test_text])
pred = loaded_model.predict(test_vec)[0]
proba = loaded_model.predict_proba(test_vec)[0]

print(f"Texte : {test_text}")
print(f"Prédiction : {'Dépressif' if pred==1 else 'Non dépressif'}")
print(f"Probabilité dépressif : {proba[1]:.2%}")

Texte : I feel very unhappy and hopeless about my life
Prédiction : Dépressif
Probabilité dépressif : 99.59%


Points forts :

Recall 91% sur dépressifs : détecte 9/10 posts dépressifs (crucial en santé mentale)

Precision 76% : quand il dit "dépressif", c’est juste 3 fois sur 4

F1 83% : excellent compromis